# 30 - Model-Level Transcriptome–Phenotype Integration

## Objective

Construct the modeling-ready cohort by integrating harmonized transcriptomic profiles with the selected pharmacological phenotype representation.

This notebook serves as the bridge between the transcriptomic layer and the pharmacogenomic layer, generating the final analysis cohort used in downstream program-discovery workflows.

## Inputs

### Harmonized cohort

- `22_integrated_modeling_cohort.csv`

Provides the unified modeling universe and lineage annotations generated during the cohort harmonization phase.

### Transcriptomic layer

- `23_expression_harmonized.parquet`

Provides harmonized gene-expression profiles for all eligible models in the integrated cohort.

### Pharmacological layer

- `26_integrated_pharmacology_cohort.csv`

Provides harmonized drug-response measurements across the integrated pharmacogenomic cohort.

## Outputs

- Model-level pharmacological phenotype table
- Integrated transcriptome–phenotype cohort
- Integration audit report
- Coverage summaries
- Modeling-ready matrices

## Key Questions

- How many models contain both transcriptomic and pharmacological information?
- What is the final overlap between the molecular and pharmacological layers?
- Is integration coverage balanced across lineages?
- Are there lineage-specific losses during integration?
- What is the final modeling universe available for downstream program-discovery analyses?

---

In [2]:
# =============================================================================
# Imports
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# =============================================================================
# Project paths
# =============================================================================

PROJECT_ROOT = Path.cwd().parents[1]

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

COHORT_PATH = INTERIM_DIR / "22_integrated_modeling_cohort.csv"

EXPRESSION_PATH = INTERIM_DIR / "23_expression_harmonized.parquet"

PHARMACOLOGY_PATH = INTERIM_DIR / "26_integrated_pharmacology_cohort.csv"

In [6]:
# =============================================================================
# Load inputs
# =============================================================================

cohort = pd.read_csv(COHORT_PATH)

expression = pd.read_parquet(EXPRESSION_PATH)

pharmacology = pd.read_csv(PHARMACOLOGY_PATH)

print("=" * 20)
print("COHORT")
print("=" * 20)
print(cohort.shape)

print("\n" + "=" * 20)
print("EXPRESSION")
print("=" * 20)
print(expression.shape)

print("\n" + "=" * 20)
print("PHARMACOLOGY")
print("=" * 20)
print(pharmacology.shape)

COHORT
(713, 8)

EXPRESSION
(713, 19194)

PHARMACOLOGY
(713, 8)


## Cohort Consistency Assessment

Before constructing the integrated analysis cohort, we verify that all input layers contain the same model universe and that no model-level mismatches were introduced during previous harmonization steps.

Because transcriptomic and pharmacological analyses will be linked at the model level, identifier consistency is a mandatory prerequisite for all downstream analyses.

In [7]:
# =============================================================================
# ModelID consistency assessment
# =============================================================================

cohort_ids = set(cohort["ModelID"])

expression_ids = set(expression["ModelID"])

pharmacology_ids = set(pharmacology["ModelID"])

print(f"Cohort models:       {len(cohort_ids):,}")
print(f"Expression models:   {len(expression_ids):,}")
print(f"Pharmacology models: {len(pharmacology_ids):,}")

print("\nOverlap checks")
print("-" * 40)

print(
    "Cohort == Expression:",
    cohort_ids == expression_ids
)

print(
    "Cohort == Pharmacology:",
    cohort_ids == pharmacology_ids
)

print(
    "Expression == Pharmacology:",
    expression_ids == pharmacology_ids
)

Cohort models:       713
Expression models:   713
Pharmacology models: 713

Overlap checks
----------------------------------------
Cohort == Expression: True
Cohort == Pharmacology: True
Expression == Pharmacology: True


In [8]:
pharmacology.columns.tolist()

['ModelID',
 'SangerModelID',
 'COSMICID',
 'CellLineName',
 'OncotreeLineage',
 'OncotreePrimaryDisease',
 'OncotreeSubtype',
 'CCLEName']

In [9]:
cohort.equals(pharmacology)

True

In [10]:
from pathlib import Path

for f in sorted(INTERIM_DIR.glob("*")):
    print(f.name)

.gitkeep
20_shared_depmap_gdsc_models.csv
21_harmonized_model_universe.csv
22_cohort_summary.csv
22_integrated_modeling_cohort.csv
22_model_layer_coverage.csv
23_expression_harmonized.parquet
24_expression_qc_summary.csv
24_gene_variability_summary.csv
26_drug_metadata.csv
26_gdsc_drug_response.csv
26_integrated_pharmacology_cohort.csv


In [11]:
drug_response = pd.read_csv(
    INTERIM_DIR / "26_gdsc_drug_response.csv"
)

drug_response.shape

(242036, 9)

In [12]:
drug_response.columns.tolist()

['SANGER_MODEL_ID',
 'DRUG_ID',
 'DRUG_NAME',
 'PUTATIVE_TARGET',
 'PATHWAY_NAME',
 'LN_IC50',
 'AUC',
 'Z_SCORE',
 'RMSE']

## Phenotype Construction

Generate the final model-level pharmacological phenotype table using the selected primary phenotype representation (raw LN_IC50).

In [ ]:
# =============================================================================
# Construct model-level phenotype table
# =============================================================================

phenotype = (
    pharmacology
    .groupby("ModelID", as_index=False)
    .agg(
        mean_ln_ic50=("LN_IC50", "mean"),
        median_ln_ic50=("LN_IC50", "median"),
        n_drugs=("DRUG_NAME", "nunique")
    )
)

phenotype.head()